# Benchmark Model Training (Self-Contained)

This notebook trains and evaluates baseline models for emotion classification.

**Objective**: Establish performance baseline with simple models - a minimum viable product.

**Data Source**: Loads `Combined_Emotion_Data.csv` directly (no need to run notebooks 05 or 09)

**Models**:
1. **Logistic Regression** - Simple linear baseline
2. **XGBoost** - Non-linear tree-based baseline

**Evaluation**: Focus on **Macro F1-score** due to class imbalance.

## Setup and Imports

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
from time import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

print(f"SageMaker version: {sagemaker.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Initialize SageMaker session and S3 client
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
s3_client = boto3.client('s3', region_name=region)

print(f"Region: {region}")
print(f"Bucket: {bucket}")
print(f"Role: {role}")

In [ ]:
# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# S3 paths for saving models
s3_benchmarks_prefix = f"s3://{bucket}/models/benchmarks"

print(f"Benchmarks will be saved to: {s3_benchmarks_prefix}")

## Load and Prepare Data

In [ ]:
# Load Combined_Emotion_Data.csv
local_file = "Datasets/Combined_Emotion_Data.csv"

if not os.path.exists(local_file):
    print(f"[ERROR] File not found: {local_file}")
    print("\nPlease ensure Combined_Emotion_Data.csv exists in the Datasets/ directory")
    raise FileNotFoundError(f"Missing required file: {local_file}")

print(f"Loading data from {local_file}...")
df = pd.read_csv(local_file)

print(f"\n[OK] Data loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Check data quality
print("Data Quality Check:\n")

# Missing values
missing = df.isnull().sum()
print(f"Missing values: {missing.sum()} total")
if missing.sum() > 0:
    print(missing[missing > 0])

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Label distribution
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

## Data Preprocessing

In [ ]:
# Standardize labels to title case (fixes 'sad' vs 'Sad' issue)
df['label'] = df['label'].str.title()

print("Labels after standardization:")
print(df['label'].value_counts().sort_index())
print(f"\nTotal samples: {len(df)}")

In [ ]:
# Define feature columns (20 acoustic features from the dataset)
feature_cols = [
    'meanfreq', 'sd', 'median', 'q25', 'q75', 'iqr', 'skew', 'kurt',
    'sp_ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun',
    'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx'
]

print(f"Using {len(feature_cols)} acoustic features:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

## Label Encoding

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

print("Label encoding:")
for label, code in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    count = (df['label'] == label).sum()
    print(f"  {code}: {label:12s} ({count:5d} samples, {count/len(df)*100:5.2f}%)")

## Train/Test Split (80/20)

In [ ]:
# Stratified split to maintain class distribution
print("Performing train/test split...\n")

X = df[feature_cols]
y = df['label_encoded']

print(f"Before split: X shape = {X.shape}, y shape = {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print(f"\nTrain/Test Split:")
print(f"  Training: {len(X_train):5d} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"  Test:     {len(X_test):5d} samples ({len(X_test)/len(df)*100:.1f}%)")
print(f"  Total:    {len(df):5d} samples")

In [ ]:
# Apply StandardScaler to normalize features
scaler = StandardScaler()

# Fit on training data only, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied (StandardScaler)")
print(f"  Mean ≈ 0, Std ≈ 1 for all features")
print(f"\nScaled training data shape: {X_train_scaled.shape}")

In [ ]:
# Verify stratification - class distribution should be same in train/test
print("Class distribution verification:\n")

train_dist = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_dist = pd.Series(y_test).value_counts(normalize=True).sort_index()

print("Training set:")
for class_id in range(len(label_encoder.classes_)):
    emotion = label_encoder.classes_[class_id]
    train_pct = train_dist.get(class_id, 0) * 100
    test_pct = test_dist.get(class_id, 0) * 100
    print(f"  {class_id}: {emotion:12s} - Train: {train_pct:5.2f}%, Test: {test_pct:5.2f}%")

# Calculate class imbalance ratio
class_counts = pd.Series(y_train).value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}x")
print("[NOTE] Using balanced class weights and Macro F1-score to handle imbalance")

## Define Evaluation Functions

In [ ]:
def evaluate_model(y_true, y_pred, model_name, label_encoder):
    """
    Comprehensive evaluation for multiclass classification.
    """
    metrics = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro'),
        'f1_per_class': f1_score(y_true, y_pred, average=None).tolist()
    }
    report = classification_report(y_true, y_pred, target_names=label_encoder.classes_, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)
    return metrics, report, cm

def plot_confusion_matrix(cm, class_names, title, save_path=None):
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

def print_metrics(metrics):
    print(f"\n{'='*60}")
    print(f"Model: {metrics['model_name']}")
    print(f"{'='*60}")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  F1 (Macro):        {metrics['f1_macro']:.4f} <- PRIMARY METRIC")
    print(f"  F1 (Weighted):     {metrics['f1_weighted']:.4f}")
    print(f"  Precision (Macro): {metrics['precision_macro']:.4f}")
    print(f"  Recall (Macro):    {metrics['recall_macro']:.4f}")
    print(f"\nPer-Class F1 Scores:")
    for i, (class_name, f1) in enumerate(zip(label_encoder.classes_, metrics['f1_per_class'])):
        print(f"  {i}: {class_name:12s} - {f1:.4f}")
    print(f"{'='*60}")

## Model 1: Logistic Regression

In [ ]:
print("Training Logistic Regression (One-vs-Rest with balanced class weights)...\n")

start_time = time()

lr_model = LogisticRegression(
    multi_class='ovr',
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

lr_model.fit(X_train_scaled, y_train)

lr_train_time = time() - start_time
print(f"[OK] Logistic Regression trained in {lr_train_time:.2f} seconds")

In [ ]:
# Make predictions on test set
start_time = time()
y_pred_lr = lr_model.predict(X_test_scaled)
lr_inference_time = time() - start_time

print(f"Predictions completed in {lr_inference_time:.4f} seconds")

In [ ]:
# Evaluate Logistic Regression
lr_metrics, lr_report, lr_cm = evaluate_model(y_test, y_pred_lr, 'Logistic Regression', label_encoder)
lr_metrics['train_time_sec'] = lr_train_time
lr_metrics['inference_time_sec'] = lr_inference_time

print_metrics(lr_metrics)

In [ ]:
# Plot confusion matrix for Logistic Regression
plot_confusion_matrix(lr_cm, label_encoder.classes_, 
                      'Logistic Regression - Confusion Matrix (Normalized)',
                      save_path='/tmp/lr_confusion_matrix.png')

## Model 2: XGBoost

In [ ]:
print("Training XGBoost Classifier...\n")

start_time = time()

# Calculate sample weights for class imbalance
sample_weights = compute_sample_weight('balanced', y_train)

xgb_model = XGBClassifier(
    max_depth=6,
    n_estimators=100,
    learning_rate=0.1,
    eval_metric='mlogloss',
    random_state=RANDOM_SEED,
    use_label_encoder=False,
    n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights, verbose=False)

xgb_train_time = time() - start_time
print(f"[OK] XGBoost trained in {xgb_train_time:.2f} seconds")

In [ ]:
# Make predictions on test set
start_time = time()
y_pred_xgb = xgb_model.predict(X_test_scaled)
xgb_inference_time = time() - start_time

print(f"Predictions completed in {xgb_inference_time:.4f} seconds")

In [ ]:
# Evaluate XGBoost
xgb_metrics, xgb_report, xgb_cm = evaluate_model(y_test, y_pred_xgb, 'XGBoost', label_encoder)
xgb_metrics['train_time_sec'] = xgb_train_time
xgb_metrics['inference_time_sec'] = xgb_inference_time

print_metrics(xgb_metrics)

In [ ]:
# Plot confusion matrix for XGBoost
plot_confusion_matrix(xgb_cm, label_encoder.classes_, 
                      'XGBoost - Confusion Matrix (Normalized)',
                      save_path='/tmp/xgb_confusion_matrix.png')

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Feature Importances:")
print(feature_importance.head(10).to_string(index=False))

## Model Comparison

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (Macro)', 'F1 (Weighted)', 'Training Time (s)'],
    'Logistic Regression': [
        lr_metrics['accuracy'],
        lr_metrics['f1_macro'],
        lr_metrics['f1_weighted'],
        lr_metrics['train_time_sec']
    ],
    'XGBoost': [
        xgb_metrics['accuracy'],
        xgb_metrics['f1_macro'],
        xgb_metrics['f1_weighted'],
        xgb_metrics['train_time_sec']
    ]
})

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

## Save Models to S3

In [ ]:
print("Saving models and artifacts to S3...\n")

# Save Logistic Regression
with open('/tmp/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
s3_client.upload_file('/tmp/lr_model.pkl', bucket, 'models/benchmarks/logistic_regression/model.pkl')
print(f"  LR Model: s3://{bucket}/models/benchmarks/logistic_regression/model.pkl")

with open('/tmp/lr_metrics.json', 'w') as f:
    json.dump(lr_metrics, f, indent=2)
s3_client.upload_file('/tmp/lr_metrics.json', bucket, 'models/benchmarks/logistic_regression/metrics.json')
print(f"  LR Metrics: s3://{bucket}/models/benchmarks/logistic_regression/metrics.json")

s3_client.upload_file('/tmp/lr_confusion_matrix.png', bucket, 'models/benchmarks/logistic_regression/confusion_matrix.png')

# Save XGBoost
with open('/tmp/xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
s3_client.upload_file('/tmp/xgb_model.pkl', bucket, 'models/benchmarks/xgboost/model.pkl')
print(f"  XGB Model: s3://{bucket}/models/benchmarks/xgboost/model.pkl")

with open('/tmp/xgb_metrics.json', 'w') as f:
    json.dump(xgb_metrics, f, indent=2)
s3_client.upload_file('/tmp/xgb_metrics.json', bucket, 'models/benchmarks/xgboost/metrics.json')
print(f"  XGB Metrics: s3://{bucket}/models/benchmarks/xgboost/metrics.json")

s3_client.upload_file('/tmp/xgb_confusion_matrix.png', bucket, 'models/benchmarks/xgboost/confusion_matrix.png')

# Save comparison
best_model = 'xgboost' if xgb_metrics['f1_macro'] > lr_metrics['f1_macro'] else 'logistic_regression'
comparison = {
    'timestamp': datetime.now().isoformat(),
    'best_model': best_model,
    'best_f1_macro': max(lr_metrics['f1_macro'], xgb_metrics['f1_macro']),
    'models': {'logistic_regression': lr_metrics, 'xgboost': xgb_metrics}
}
with open('/tmp/benchmark_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)
s3_client.upload_file('/tmp/benchmark_comparison.json', bucket, 'models/benchmarks/benchmark_comparison.json')

print("\n[OK] All artifacts saved to S3")

## Summary

In [ ]:
print("="*70)
print("BENCHMARK MODEL TRAINING - COMPLETE")
print("="*70)

print(f"\nDataset: {len(df):,} samples, {len(feature_cols)} features, {len(label_encoder.classes_)} classes")
print(f"Train/Test: {len(X_train):,} / {len(X_test):,}")

print(f"\nResults:")
print(f"  Logistic Regression: F1 (Macro) = {lr_metrics['f1_macro']:.4f}")
print(f"  XGBoost:             F1 (Macro) = {xgb_metrics['f1_macro']:.4f}")

print(f"\nBest Model: {best_model.replace('_', ' ').title()}")
print(f"\nModels saved to: s3://{bucket}/models/benchmarks/")
print("="*70)

## Release Resources

In [ ]:
%%html
<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>